<a href="https://colab.research.google.com/github/BilalKhanAI/Dubai-Real-Estate-Price-Prediction/blob/main/Dubai_Housing_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# 1. Load dataset (change file name if yours is named differently)
df = pd.read_csv("dubai_properties.csv")

print("Dataset loaded! Shape:", df.shape)
df.head()

In [ ]:
# Keep essential columns (edit list based on your dataset columns)
features = ["size_sqft", "bedrooms", "bathrooms", "location"]
target = "price"

# Drop missing target values
df = df.dropna(subset=[target])

# Clean numeric features
for col in ["size_sqft", "bedrooms", "bathrooms"]:
  if col in df.columns:
    df[col] = df[col].fillna(df[col].median())

# Convert locations to numerical values (One-Hot Encoding)
df_clean = pd.get_dummies(df[features + [target]], drop_first=True)

print("Data cleaned successfully!")

In [ ]:
# Separate features (X) and target (y)
X = df_clean.drop(columns=[target])
y = df_clean[target]

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predictions & Metrics
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"--- MODEL RESULTS ---")
print(f"Mean Absolute Error (MAE): AED {mae:,.2f}")
print(f"Model Accuracy (R² Score): {r2 * 100:.2f}%")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test / 1e6, predictions / 1e6, alpha=0.5, color="#1f77b4")
plt.plot(
    [y_test.min() / 1e6, y_test.max() / 1e6],
    [y_test.min() / 1e6, y_test.max() / 1e6],
    "r--",
    lw=2,
)
plt.xlabel("Actual Price (Millions AED)")
plt.ylabel("Predicted Price (Millions AED)")
plt.title("Actual vs. Predicted Property Prices in Dubai")
plt.grid(True)
plt.savefig("actual_vs_predicted.png", dpi=300)
plt.show()